
# PHY5003 — Assessment 1 Draft Notebook  
## QAOA (Max-Cut) implemented in **Qiskit** and **Q#**

**Topic:** Quantum Approximate Optimization Algorithm (QAOA) for Max-Cut

---

## What this notebook covers

This notebook is designed to support the assignment requirements:

- explain the **Max-Cut** problem from the beginning
- show a **classical Python** solution
- implement **QAOA in Qiskit**
- include a **Q# implementation**
- export the circuit to **OpenQASM 3**
- explain the OpenQASM instructions line by line
- compare classical and quantum approaches
- discuss the conditions needed for **quantum advantage / quantum supremacy**

---

## Important note

This notebook uses **Qiskit** for the executable quantum workflow and **OpenQASM 3 export**.  
The Q# section is included as a notebook-friendly code path and as a standalone `.qs` file.  
If your environment does not have the required packages installed yet, run the setup cell below first.


In [ ]:

# Run this once in a fresh environment if needed.
# In Jupyter, uncomment and execute:

# %pip install qiskit matplotlib networkx numpy
# %pip install "qdk[jupyter]"

print("Setup cell ready. Uncomment the install commands if your environment is missing packages.")


In [ ]:

import itertools
import time
from collections import Counter

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    from qiskit import qasm3
    HAVE_QISKIT = True
except Exception as exc:
    HAVE_QISKIT = False
    QISKIT_IMPORT_ERROR = exc
    print("Qiskit import failed:", exc)

print("Qiskit available:", HAVE_QISKIT)



# 1. The problem: what is Max-Cut?

A graph has:

- **nodes** (the objects)
- **edges** (the relationships that matter)

For Max-Cut, each node is assigned to **one of two groups**:

- group 0
- group 1

An edge is said to be **cut** if its endpoints end up in different groups.

For an edge $(u,v)$:

- if $x_u \neq x_v$, that edge contributes **1**
- if $x_u = x_v$, that edge contributes **0**

That is why the name is **Max-Cut**:
we want the **maximum number of existing edges** to cross the split between the two groups.


In [ ]:

# Small graph used throughout the notebook.
# It is deliberately small enough to solve exactly with brute force.
# Edges: square + one diagonal.

NUM_NODES = 4
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (0, 2)]

G = nx.Graph()
G.add_nodes_from(range(NUM_NODES))
G.add_edges_from(EDGES)

pos = nx.circular_layout(G)
plt.figure(figsize=(5, 5))
nx.draw(
    G,
    pos,
    with_labels=True,
    node_color="lightgray",
    node_size=1200,
    font_size=12,
)
plt.title("Max-Cut example graph")
plt.show()

print("Nodes:", list(G.nodes()))
print("Edges:", list(G.edges()))



# 2. Classical solution

A classical exact solution checks every possible grouping.

With `n` nodes there are `2^n` possible binary assignments, because each node can be in:

- group 0, or
- group 1

That is why the exact classical approach becomes expensive quickly as the graph grows.


In [ ]:

def cut_value(bits, edges):
    """Return the number of cut edges for a given bit assignment."""
    return sum(int(bits[u] != bits[v]) for u, v in edges)

def brute_force_max_cut(num_nodes, edges):
    """Enumerate all bitstrings and return the maximum cut value and best assignments."""
    best_value = -1
    best_assignments = []

    for bits in itertools.product([0, 1], repeat=num_nodes):
        value = cut_value(bits, edges)
        if value > best_value:
            best_value = value
            best_assignments = [bits]
        elif value == best_value:
            best_assignments.append(bits)

    return best_value, best_assignments

classical_best_value, classical_best_assignments = brute_force_max_cut(NUM_NODES, EDGES)
classical_best_value, classical_best_assignments[:10]


In [ ]:

print("Classical optimum cut value:", classical_best_value)
print("Best bitstrings:")
for bits in classical_best_assignments:
    print(bits)


In [ ]:

def plot_partition(graph, bits, title):
    colors = ["tab:blue" if bit == 0 else "tab:orange" for bit in bits]
    edge_colors = ["tab:red" if bits[u] != bits[v] else "black" for u, v in graph.edges()]
    plt.figure(figsize=(5, 5))
    nx.draw(
        graph,
        pos,
        with_labels=True,
        node_color=colors,
        edge_color=edge_colors,
        node_size=1200,
        width=2,
        font_size=12,
    )
    plt.title(title)
    plt.show()

plot_partition(G, classical_best_assignments[0], f"One optimal cut, value = {classical_best_value}")



# 3. Why use QAOA?

**QAOA** stands for **Quantum Approximate Optimization Algorithm**.

- **Max-Cut** is the problem.
- **QAOA** is the quantum algorithm used to search for a good solution.

QAOA is a **hybrid quantum-classical algorithm**:

1. a quantum circuit prepares and transforms a superposition of candidate solutions
2. a classical optimizer updates the circuit parameters
3. the loop repeats until good parameters are found

For Max-Cut, each node is represented by **one qubit**.

A bitstring like `0101` means:

- node 0 → group 0
- node 1 → group 1
- node 2 → group 0
- node 3 → group 1

So each computational basis state corresponds to one possible cut.

---

## Cost Hamiltonian for Max-Cut

For one edge $(u,v)$, a standard Max-Cut cost term is

$
C_{uv} = \frac{1 - Z_u Z_v}{2}
$

This gives:

- value `1` when the two nodes are in different groups
- value `0` when they are in the same group

The full cost Hamiltonian is

$
C = \sum_{(u,v)\in E} \frac{1 - Z_u Z_v}{2}
$

QAOA alternates between:

- a **cost unitary** based on this Hamiltonian
- a **mixer unitary** that explores other bitstrings

For one QAOA layer (`p = 1`), the state is:

$
|\psi(\gamma,\beta)\rangle = U_B(\beta)\,U_C(\gamma)\,|+\rangle^{\otimes n}
$

where:

$
U_C(\gamma) = e^{-i\gamma C}
\quad\text{and}\quad
U_B(\beta) = e^{-i\beta \sum_i X_i}
$

For this notebook we use a simple, transparent `p = 1` implementation.


In [ ]:

def qaoa_maxcut_circuit(num_nodes, edges, gamma, beta, measure=False):
    """Build a p=1 QAOA circuit for Max-Cut using an explicit CX-RZ-CX cost layer."""
    if not HAVE_QISKIT:
        raise ImportError(f"Qiskit is required. Original import error: {QISKIT_IMPORT_ERROR}")

    qc = QuantumCircuit(num_nodes, num_nodes if measure else 0)

    # Initial equal superposition over all bitstrings
    for q in range(num_nodes):
        qc.h(q)

    # Cost unitary:
    # For each edge (u, v), implement exp(-i * gamma * (I - Z_u Z_v)/2).
    # Up to a global phase, this is equivalent to RZZ(-gamma).
    # We decompose RZZ(-gamma) as CX(u,v) -> RZ(-gamma) on v -> CX(u,v).
    for u, v in edges:
        qc.cx(u, v)
        qc.rz(-gamma, v)
        qc.cx(u, v)

    # Mixer unitary: exp(-i * beta * X) = RX(2 * beta)
    for q in range(num_nodes):
        qc.rx(2 * beta, q)

    if measure:
        qc.measure(range(num_nodes), range(num_nodes))

    return qc


In [ ]:

if HAVE_QISKIT:
    demo_qc = qaoa_maxcut_circuit(NUM_NODES, EDGES, gamma=0.7, beta=0.4, measure=True)
    print(demo_qc.draw(output="text"))
else:
    print("Install Qiskit, then rerun this cell.")



# 4. Evaluate QAOA without a simulator backend

To keep the notebook lightweight, the Qiskit part uses `Statevector` instead of requiring Aer.

That lets us:

- build the QAOA circuit
- evaluate probabilities
- estimate the expected cut value
- sample bitstrings from the final state


In [ ]:

def expected_cut_from_probabilities(probabilities, num_nodes, edges):
    total = 0.0
    for index, prob in enumerate(probabilities):
        # Qiskit basis-state ordering uses little-endian qubit convention internally.
        # Reversing the formatted bitstring makes node index 0 correspond to the first character.
        bitstring = format(index, f"0{num_nodes}b")[::-1]
        bits = tuple(int(b) for b in bitstring)
        total += prob * cut_value(bits, edges)
    return total

def qaoa_expected_cut_value(num_nodes, edges, gamma, beta):
    qc = qaoa_maxcut_circuit(num_nodes, edges, gamma, beta, measure=False)
    state = Statevector.from_instruction(qc)
    probabilities = np.abs(state.data) ** 2
    return expected_cut_from_probabilities(probabilities, num_nodes, edges), state

if HAVE_QISKIT:
    exp_value, state = qaoa_expected_cut_value(NUM_NODES, EDGES, gamma=0.7, beta=0.4)
    print("Expected cut value:", exp_value)


In [ ]:

# Coarse parameter search.
# This is intentionally simple and transparent for an assignment notebook.

def grid_search_qaoa(num_nodes, edges, gamma_values, beta_values):
    best = {
        "gamma": None,
        "beta": None,
        "expected_cut": -1.0,
        "state": None,
    }

    for gamma in gamma_values:
        for beta in beta_values:
            exp_cut, state = qaoa_expected_cut_value(num_nodes, edges, gamma, beta)
            if exp_cut > best["expected_cut"]:
                best = {
                    "gamma": gamma,
                    "beta": beta,
                    "expected_cut": exp_cut,
                    "state": state,
                }
    return best

if HAVE_QISKIT:
    gamma_grid = np.linspace(0, np.pi, 31)
    beta_grid = np.linspace(0, np.pi / 2, 31)

    start = time.perf_counter()
    best_qaoa = grid_search_qaoa(NUM_NODES, EDGES, gamma_grid, beta_grid)
    qaoa_search_time = time.perf_counter() - start

    best_qaoa
else:
    print("Install Qiskit, then rerun this cell.")


In [ ]:

if HAVE_QISKIT:
    print("Best gamma:", best_qaoa["gamma"])
    print("Best beta:", best_qaoa["beta"])
    print("Best expected cut:", best_qaoa["expected_cut"])
    print("Classical optimum:", classical_best_value)
    print("Grid-search runtime (s):", qaoa_search_time)


In [ ]:

def sample_qaoa_counts(num_nodes, edges, gamma, beta, shots=2048):
    qc = qaoa_maxcut_circuit(num_nodes, edges, gamma, beta, measure=False)
    state = Statevector.from_instruction(qc)
    raw_counts = state.sample_counts(shots=shots)

    # Convert strings to node-order convention used elsewhere in this notebook.
    converted = Counter()
    for qiskit_bitstring, count in raw_counts.items():
        converted[qiskit_bitstring[::-1]] += count
    return converted

if HAVE_QISKIT:
    qaoa_counts = sample_qaoa_counts(
        NUM_NODES,
        EDGES,
        best_qaoa["gamma"],
        best_qaoa["beta"],
        shots=2048,
    )
    print(qaoa_counts.most_common(10))


In [ ]:

if HAVE_QISKIT:
    top_bitstring, top_count = qaoa_counts.most_common(1)[0]
    top_bits = tuple(int(b) for b in top_bitstring)

    print("Most common sampled bitstring:", top_bitstring)
    print("Cut value of most common sampled bitstring:", cut_value(top_bits, EDGES))

    plot_partition(G, top_bits, f"Most common QAOA bitstring {top_bitstring}, cut = {cut_value(top_bits, EDGES)}")



# 5. OpenQASM 3 export

We export the measured QAOA circuit to **OpenQASM 3**.


In [44]:

if HAVE_QISKIT:
    measured_qc = qaoa_maxcut_circuit(
        NUM_NODES,
        EDGES,
        best_qaoa["gamma"],
        best_qaoa["beta"],
        measure=True,
    )
    openqasm_text = qasm3.dumps(measured_qc)
    print(openqasm_text)


OPENQASM 3.0;
include "stdgates.inc";
bit[4] c;
qubit[4] q;
h q[0];
h q[1];
h q[2];
h q[3];
cx q[0], q[1];
rz(-pi/6) q[1];
cx q[0], q[1];
cx q[1], q[2];
rz(-pi/6) q[2];
cx q[1], q[2];
cx q[2], q[3];
rz(-pi/6) q[3];
cx q[2], q[3];
cx q[3], q[0];
rz(-pi/6) q[0];
cx q[3], q[0];
cx q[0], q[2];
rz(-pi/6) q[2];
cx q[0], q[2];
rx(pi/5) q[0];
rx(pi/5) q[1];
rx(pi/5) q[2];
rx(pi/5) q[3];
c[0] = measure q[0];
c[1] = measure q[1];
c[2] = measure q[2];
c[3] = measure q[3];




# 6. Q# implementation

This section provides the same small Max-Cut QAOA idea in **Q#**.

## Practical note

Modern notebook use for Q# relies on the **QDK Jupyter flow**.
If you installed `qdk[jupyter]`, importing `qsharp` should enable `%%qsharp` cells.

If your notebook environment does not support `%%qsharp`, you can still:

- copy the Q# code into `qsharp/QAOAMaxCut.qs`
- run it from VS Code with the QDK extension
- keep the notebook as the explanatory document and the `.qs` file as the second-language implementation


In [43]:

qsharp_source = r'''
import Std.Intrinsic.*;
import Std.Measurement.*;
import Std.Arrays.*;

operation RunQAOAMaxCut(gamma : Double, beta : Double) : Result[] {
    // Same 4-node graph used in the notebook:
    // edges = [(0,1), (1,2), (2,3), (3,0), (0,2)]
    use qubits = Qubit[4];

    // Initial |+>^n state
    for q in qubits {
        H(q);
    }

    // Cost layer using CX-Rz-CX for each edge.
    // Up to a global phase, this corresponds to the Max-Cut cost unitary.
    CNOT(qubits[0], qubits[1]);
    Rz(-gamma, qubits[1]);
    CNOT(qubits[0], qubits[1]);

    CNOT(qubits[1], qubits[2]);
    Rz(-gamma, qubits[2]);
    CNOT(qubits[1], qubits[2]);

    CNOT(qubits[2], qubits[3]);
    Rz(-gamma, qubits[3]);
    CNOT(qubits[2], qubits[3]);

    CNOT(qubits[3], qubits[0]);
    Rz(-gamma, qubits[0]);
    CNOT(qubits[3], qubits[0]);

    CNOT(qubits[0], qubits[2]);
    Rz(-gamma, qubits[2]);
    CNOT(qubits[0], qubits[2]);

    // Mixer layer
    for q in qubits {
        Rx(2.0 * beta, q);
    }

    let results = MeasureEachZ(qubits);
    ResetAll(qubits);
    return results;
}
'''.strip()

print(qsharp_source)


import Std.Intrinsic.*;
import Std.Measurement.*;
import Std.Arrays.*;

operation RunQAOAMaxCut(gamma : Double, beta : Double) : Result[] {
    // Same 4-node graph used in the notebook:
    // edges = [(0,1), (1,2), (2,3), (3,0), (0,2)]
    use qubits = Qubit[4];

    // Initial |+>^n state
    for q in qubits {
        H(q);
    }

    // Cost layer using CX-Rz-CX for each edge.
    // Up to a global phase, this corresponds to the Max-Cut cost unitary.
    CNOT(qubits[0], qubits[1]);
    Rz(-gamma, qubits[1]);
    CNOT(qubits[0], qubits[1]);

    CNOT(qubits[1], qubits[2]);
    Rz(-gamma, qubits[2]);
    CNOT(qubits[1], qubits[2]);

    CNOT(qubits[2], qubits[3]);
    Rz(-gamma, qubits[3]);
    CNOT(qubits[2], qubits[3]);

    CNOT(qubits[3], qubits[0]);
    Rz(-gamma, qubits[0]);
    CNOT(qubits[3], qubits[0]);

    CNOT(qubits[0], qubits[2]);
    Rz(-gamma, qubits[2]);
    CNOT(qubits[0], qubits[2]);

    // Mixer layer
    for q in qubits {
        Rx(2.0 * beta, q);
    }

    le

In [42]:

# Optional execution path if the QDK is installed.
# This is intentionally defensive because many Jupyter environments will not have qsharp configured yet.

try:
    from qdk import qsharp
    HAVE_QSHARP = True
except Exception as exc:
    HAVE_QSHARP = False
    QSHARP_IMPORT_ERROR = exc
    print("Q# support not available in this notebook environment:", exc)

print("Q# available:", HAVE_QSHARP)


Q# available: True


In [41]:

# Optional: define the Q# operation in the notebook and run local shots.
# If your environment does not support qsharp, skip this cell and use the standalone .qs file instead.

if HAVE_QSHARP:
    qsharp.eval(qsharp_source)

    gamma = float(best_qaoa["gamma"]) if HAVE_QISKIT else 0.7
    beta = float(best_qaoa["beta"]) if HAVE_QISKIT else 0.4

    try:
        qsharp_results = qsharp.run(f"RunQAOAMaxCut({gamma}, {beta})", shots=512)
        print(qsharp_results)
    except Exception as exc:
        print("Q# execution failed in this environment:", exc)
        print("Use the standalone qsharp/QAOAMaxCut.qs file in VS Code with the QDK extension.")


[[Zero, One, Zero, One], [One, Zero, Zero, One], [One, Zero, Zero, Zero], [One, Zero, Zero, Zero], [One, Zero, One, Zero], [One, Zero, Zero, Zero], [One, Zero, One, One], [One, Zero, Zero, One], [Zero, One, One, One], [Zero, One, One, One], [Zero, Zero, One, One], [One, Zero, Zero, One], [Zero, One, Zero, One], [Zero, Zero, One, One], [One, One, Zero, One], [One, One, Zero, Zero], [Zero, Zero, One, Zero], [Zero, One, Zero, One], [One, One, Zero, One], [One, Zero, One, Zero], [One, Zero, Zero, One], [Zero, One, One, One], [Zero, One, Zero, One], [One, One, Zero, One], [One, Zero, One, Zero], [Zero, One, One, One], [Zero, One, Zero, One], [Zero, Zero, One, Zero], [One, One, One, Zero], [Zero, One, One, Zero], [One, Zero, One, Zero], [Zero, Zero, One, One], [Zero, One, Zero, One], [One, Zero, Zero, One], [One, Zero, Zero, One], [Zero, Zero, One, One], [One, Zero, One, Zero], [One, Zero, Zero, One], [One, One, Zero, Zero], [Zero, One, One, One], [One, Zero, Zero, One], [One, Zero, Zero, On


# 7. Classical vs quantum comparison

For this tiny graph, the classical exact solution is easy.

That is expected.  
A small example like this is for **understanding and verification**, not for proving practical quantum advantage.


In [ ]:

# Tiny timing comparison for the chosen graph.
# This compares:
# 1) exact brute-force classical search
# 2) simple coarse-grid QAOA parameter search
# The comparison is illustrative only.

start = time.perf_counter()
classical_value, classical_assignments = brute_force_max_cut(NUM_NODES, EDGES)
classical_time = time.perf_counter() - start

print("Classical brute-force optimum:", classical_value)
print("Classical runtime (s):", classical_time)

if HAVE_QISKIT:
    print("QAOA coarse-grid search expected best value:", best_qaoa["expected_cut"])
    print("QAOA coarse-grid search runtime (s):", qaoa_search_time)



## Interpretation

For this notebook's small graph:

- the **classical brute-force** method is trivial and exact
- the **QAOA** method is approximate and uses tunable parameters
- the quantum path is still useful because it demonstrates how combinatorial optimization can be encoded into a circuit

For larger hard instances, the motivation for quantum methods is that exact classical search scales exponentially.
That does **not** mean QAOA automatically wins today.
It means QAOA is a candidate approach for future larger, lower-noise quantum hardware.



# 8. Conditions needed for quantum advantage / quantum supremacy

To argue that QAOA could outperform classical methods on Max-Cut, you would need:

1. **larger problem sizes**  
   enough qubits to encode substantially larger graphs

2. **deeper QAOA circuits**  
   more layers `p`, which usually increases circuit depth and noise sensitivity

3. **low two-qubit gate error rates**  
   because QAOA cost layers rely heavily on entangling gates

4. **enough coherence time**  
   the quantum state must survive the full circuit

5. **good parameter optimization**  
   the classical outer-loop optimizer must find useful angles efficiently

6. **fair classical baselines**  
   comparisons should be against strong classical heuristics, not only brute force




# 9. Unit tests


In [ ]:

# Quick in-notebook checks

assert classical_best_value == 4, "Unexpected Max-Cut optimum for the chosen graph."

for bits in classical_best_assignments:
    assert cut_value(bits, EDGES) == classical_best_value

print("Basic classical checks passed.")

if HAVE_QISKIT:
    qc = qaoa_maxcut_circuit(NUM_NODES, EDGES, gamma=0.7, beta=0.4, measure=True)
    assert qc.num_qubits == NUM_NODES
    assert qc.num_clbits == NUM_NODES
    print("Basic Qiskit circuit checks passed.")
